# ChemAI: Predict the Cure — final solution

В этом ноутбуке собран финальный воспроизводимый pipeline для соревнования **ChemAI: Predict the Cure**.

Задача — предсказать три таргета:

- `IC50` — концентрация вещества, подавляющая 50% активности вируса;
- `CC50` — концентрация, токсичная для 50% клеток;
- `SI` — индекс селективности.

Итоговое решение состоит из трёх основных частей:

1. **Calibrated hybrid-best ветка**  
   Первая сильная ветка на основе target-wise blend, aggregation/KNN-сигнала для `SI` и калиброванной ratio-ветки для `IC50`/`CC50`.

2. **CatBoost MAE ветка**  
   Независимая CatBoost-ветка с отдельными моделями по таргетам, seed averaging и postprocessing для `SI`.

3. **Финальное объединение и postprocessing**  
   `IC50` и `CC50` берутся как blend двух веток, `SI` остаётся из CatBoost MAE. После этого применяется postprocessing только для `IC50`.

Финальный public score выбранного решения: `263.61199`.

## 1. Импорты, настройки и загрузка данных

### 1.1 Импорты и глобальные настройки

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold, KFold, StratifiedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error

warnings.filterwarnings(
    "ignore",
    message=".*does not have valid feature names.*",
    category=UserWarning,
)

RANDOM_STATE = 42
N_SPLITS = 5

DATA_DIR = Path("../data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

SUBMISSIONS_DIR = Path("../submissions")
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COLUMNS = ["IC50", "CC50", "SI"]
ID_COLUMN = "index"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6f}".format)

print("Настройки подготовлены.")

Настройки подготовлены.


### 1.2 Загрузка данных

In [2]:
train_data = pd.read_csv(TRAIN_PATH)
test_data = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Данные успешно загружены.")
print(f"Размер train_data: {train_data.shape}")
print(f"Размер test_data: {test_data.shape}")
print(f"Размер sample_submission: {sample_submission.shape}")

Данные успешно загружены.
Размер train_data: (751, 214)
Размер test_data: (250, 211)
Размер sample_submission: (250, 4)


### 1.3 Приведение названий таргетов

In [3]:
target_rename_mapping = {
    "IC50, mM": "IC50",
    "CC50, mM": "CC50",
}

train_data = train_data.rename(columns=target_rename_mapping)

missing_targets = [
    column for column in TARGET_COLUMNS
    if column not in train_data.columns
]

if missing_targets:
    raise ValueError(f"В train отсутствуют таргеты: {missing_targets}")

print("Названия таргетов приведены к единому формату.")
print(f"Таргеты: {TARGET_COLUMNS}")

Названия таргетов приведены к единому формату.
Таргеты: ['IC50', 'CC50', 'SI']


### 1.4 Подготовка признаков и целевых переменных

In [4]:
excluded_columns = TARGET_COLUMNS.copy()

if ID_COLUMN in train_data.columns:
    excluded_columns.append(ID_COLUMN)

feature_columns = [
    column for column in train_data.columns
    if column not in excluded_columns
]

test_feature_columns = [
    column for column in test_data.columns
    if column != ID_COLUMN
]

missing_features_in_test = sorted(set(feature_columns) - set(test_feature_columns))
extra_features_in_test = sorted(set(test_feature_columns) - set(feature_columns))

if missing_features_in_test:
    raise ValueError(
        "В test отсутствуют признаки из train: "
        f"{missing_features_in_test}"
    )

if extra_features_in_test:
    raise ValueError(
        "В test есть лишние признаки: "
        f"{extra_features_in_test}"
    )

X = train_data[feature_columns].copy()
y = train_data[TARGET_COLUMNS].copy()
X_test = test_data[feature_columns].copy()

print("Матрицы признаков и таргетов подготовлены.")
print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(f"X_test: {X_test.shape}")
print(f"Количество признаков: {len(feature_columns)}")

Матрицы признаков и таргетов подготовлены.
X: (751, 210)
y: (751, 3)
X_test: (250, 210)
Количество признаков: 210


### 1.5 Базовые проверки данных

In [5]:
train_missing_total = int(X.isna().sum().sum())
test_missing_total = int(X_test.isna().sum().sum())

train_infinite_total = int(np.isinf(X.to_numpy()).sum())
test_infinite_total = int(np.isinf(X_test.to_numpy()).sum())

print("Проверка данных перед обучением")
print(f"Пропусков в X: {train_missing_total}")
print(f"Пропусков в X_test: {test_missing_total}")
print(f"Бесконечных значений в X: {train_infinite_total}")
print(f"Бесконечных значений в X_test: {test_infinite_total}")

if train_infinite_total > 0 or test_infinite_total > 0:
    raise ValueError("В признаках есть бесконечные значения.")

expected_submission_columns = [ID_COLUMN] + TARGET_COLUMNS

if list(sample_submission.columns) != expected_submission_columns:
    raise ValueError(
        "Формат sample_submission не совпадает с ожидаемым: "
        f"{expected_submission_columns}"
    )

print("Формат sample_submission корректный.")

Проверка данных перед обучением
Пропусков в X: 24
Пропусков в X_test: 12
Бесконечных значений в X: 0
Бесконечных значений в X_test: 0
Формат sample_submission корректный.


## 2. Общие функции

В этом разделе собраны функции и объекты, которые используются в финальном pipeline: метрика соревнования, hash feature-векторов, схемы кросс-валидации, проверки предсказаний и сохранение submission.

### 2.1 Метрика соревнования

In [6]:
def competition_rmse(y_true, y_pred):
    """Считает RMSE отдельно по каждому таргету и усредняет результат."""
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)

    target_rmse = np.sqrt(
        np.mean((y_true_array - y_pred_array) ** 2, axis=0)
    )

    return target_rmse.mean()


def competition_rmse_by_target(y_true, y_pred, target_columns):
    """Считает RMSE отдельно для каждого таргета."""
    y_true_array = np.asarray(y_true)
    y_pred_array = np.asarray(y_pred)

    target_rmse = np.sqrt(
        np.mean((y_true_array - y_pred_array) ** 2, axis=0)
    )

    return pd.Series(target_rmse, index=target_columns)

### 2.2 Hash feature-векторов

In [7]:
def get_feature_hashes(feature_data):
    """Создаёт hash для строк с одинаковым признаковым описанием."""
    return pd.util.hash_pandas_object(
        feature_data,
        index=False,
    ).astype(str)


feature_groups = get_feature_hashes(X)

print("Группы одинаковых объектов созданы.")
print(f"Количество строк в train: {len(X)}")
print(f"Количество уникальных feature-групп: {feature_groups.nunique()}")
print(
    "Количество повторяющихся строк по признакам: "
    f"{len(X) - feature_groups.nunique()}"
)

Группы одинаковых объектов созданы.
Количество строк в train: 751
Количество уникальных feature-групп: 630
Количество повторяющихся строк по признакам: 121


### 2.3 Схемы кросс-валидации

In [8]:
standard_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

group_cv = GroupKFold(
    n_splits=N_SPLITS,
)

si_stratification_target = (y["SI"] > 8).astype(int)

mae_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Схемы кросс-валидации подготовлены.")
print(f"KFold: {N_SPLITS} folds")
print(f"GroupKFold: {N_SPLITS} folds")
print(f"StratifiedKFold для CatBoost MAE: {N_SPLITS} folds")

Схемы кросс-валидации подготовлены.
KFold: 5 folds
GroupKFold: 5 folds
StratifiedKFold для CatBoost MAE: 5 folds


### 2.4 Базовые границы для `SI`

In [9]:
SI_LOWER_BOUND = 1.0
SI_UPPER_BOUND_Q0985 = float(y["SI"].quantile(0.985))
SI_UPPER_BOUND_Q0995 = float(y["SI"].quantile(0.995))

print(f"Нижняя граница SI: {SI_LOWER_BOUND}")
print(f"Верхняя граница SI q0.985: {SI_UPPER_BOUND_Q0985:.6f}")
print(f"Верхняя граница SI q0.995: {SI_UPPER_BOUND_Q0995:.6f}")

Нижняя граница SI: 1.0
Верхняя граница SI q0.985: 635.678571
Верхняя граница SI q0.995: 4251.275000


### 2.5 Проверка и описание предсказаний

In [10]:
def validate_predictions(predictions, expected_rows, target_columns):
    """Проверяет форму, NaN и бесконечные значения в предсказаниях."""
    predictions_array = np.asarray(predictions)

    expected_shape = (expected_rows, len(target_columns))

    if predictions_array.shape != expected_shape:
        raise ValueError(
            "Некорректная форма предсказаний: "
            f"{predictions_array.shape}, ожидалось {expected_shape}."
        )

    if np.isnan(predictions_array).any():
        raise ValueError("В предсказаниях есть NaN.")

    if np.isinf(predictions_array).any():
        raise ValueError("В предсказаниях есть бесконечные значения.")

    return True


def describe_predictions(predictions, target_columns):
    """Возвращает описательную статистику по предсказаниям."""
    predictions_frame = pd.DataFrame(
        predictions,
        columns=target_columns,
    )

    return predictions_frame.describe().T

### 2.6 Создание submission

In [11]:
def create_submission(predictions, sample_submission, output_path):
    """Создаёт и сохраняет submission-файл."""
    validate_predictions(
        predictions=predictions,
        expected_rows=len(sample_submission),
        target_columns=TARGET_COLUMNS,
    )

    submission = sample_submission.copy()
    submission[TARGET_COLUMNS] = predictions

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    submission.to_csv(output_path, index=False)

    print(f"Submission сохранён: {output_path}")
    print(f"Размер submission: {submission.shape}")

    return submission

## 3. Первая ветка: calibrated hybrid-best

В этом разделе воспроизводится первая сильная ветка решения — calibrated hybrid-best.

Она состоит из нескольких выбранных компонентов:

1. target-wise blend моделей на `original` и `corr_0990` feature sets;
2. mean-aggregation pipeline для одинаковых feature-векторов;
3. KNN-сигнал для `SI`;
4. fine meta target-wise blend;
5. isotonic-калиброванная ratio-ветка для `IC50` и `CC50`.

В исследовательском ноутбуке эти компоненты подбирались поэтапно. Здесь оставлена только финальная конфигурация, которая используется дальше в итоговом pipeline.

### 3.1 Feature sets

Для первой ветки используются несколько наборов признаков. Финально в calibrated hybrid-best участвуют:

- `original` — все исходные признаки;
- `corr_0990` — признаки после удаления константных, дублирующихся и сильно коррелированных признаков.

Остальные наборы сохраняются в словаре для согласованности с исследовательским ноутбуком.

In [12]:
def get_columns_without_constants(feature_data):
    """Удаляет константные признаки."""
    return [
        column for column in feature_data.columns
        if feature_data[column].nunique(dropna=False) > 1
    ]


def get_columns_without_duplicates(feature_data):
    """Удаляет дублирующиеся признаки."""
    duplicated_columns_mask = feature_data.T.duplicated()

    return [
        column
        for column, is_duplicated in zip(
            feature_data.columns,
            duplicated_columns_mask,
        )
        if not is_duplicated
    ]


def get_columns_without_high_correlation(feature_data, threshold):
    """Удаляет один признак из пары с корреляцией выше заданного порога."""
    correlation_matrix = feature_data.corr().abs()

    upper_triangle = correlation_matrix.where(
        np.triu(
            np.ones(correlation_matrix.shape),
            k=1,
        ).astype(bool)
    )

    columns_to_drop = [
        column for column in upper_triangle.columns
        if any(upper_triangle[column] > threshold)
    ]

    selected_columns = [
        column for column in feature_data.columns
        if column not in columns_to_drop
    ]

    return selected_columns, columns_to_drop


no_constant_columns = get_columns_without_constants(X)
no_duplicate_columns = get_columns_without_duplicates(X)
no_constant_no_duplicate_columns = get_columns_without_duplicates(
    X[no_constant_columns]
)

corr_0995_columns, corr_0995_removed_columns = (
    get_columns_without_high_correlation(
        X[no_constant_no_duplicate_columns],
        threshold=0.995,
    )
)

corr_0990_columns, corr_0990_removed_columns = (
    get_columns_without_high_correlation(
        X[no_constant_no_duplicate_columns],
        threshold=0.990,
    )
)

feature_sets = {
    "original": feature_columns,
    "no_constant": no_constant_columns,
    "no_duplicate": no_duplicate_columns,
    "no_constant_no_duplicate": no_constant_no_duplicate_columns,
    "corr_0995": corr_0995_columns,
    "corr_0990": corr_0990_columns,
}

feature_set_summary = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "n_features": len(columns),
            "removed_features": len(feature_columns) - len(columns),
        }
        for feature_set_name, columns in feature_sets.items()
    ]
)

feature_set_summary

,feature_set,n_features,removed_features
0,original,210,0
1,no_constant,192,18
2,no_duplicate,187,23
3,no_constant_no_duplicate,186,24
4,corr_0995,183,27
5,corr_0990,179,31


### 3.2 Базовые модели для Direct-blend ветки

Для `IC50` и `CC50` используется `ExtraTreesRegressor`, а для `SI` берётся прогноз CatBoost. После этого `SI` ограничивается снизу `1.0` и сверху 0.985-квантилем train.

In [13]:
def make_extratrees_model(random_state):
    """Создаёт ExtraTreesRegressor для multi-target задачи."""
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    max_features=1.0,
                    random_state=random_state,
                    n_jobs=-1,
                ),
            ),
        ]
    )


def make_catboost_model(random_state):
    """Создаёт CatBoostRegressor для multi-target задачи."""
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                CatBoostRegressor(
                    iterations=800,
                    learning_rate=0.03,
                    depth=4,
                    loss_function="MultiRMSE",
                    random_seed=random_state,
                    verbose=False,
                ),
            ),
        ]
    )


def fit_direct_blend_feature_set(
    X_train_full,
    X_test_full,
    y_train_full,
    cv,
    groups=None,
):
    """Строит OOF и test-прогноз для feature set: ExtraTrees + CatBoost-SI."""
    oof_predictions = np.zeros((len(X_train_full), len(TARGET_COLUMNS)))

    for fold_number, (train_index, valid_index) in enumerate(
        cv.split(X_train_full, y_train_full, groups),
        start=1,
    ):
        print(f"Обучается fold {fold_number}")

        X_train = X_train_full.iloc[train_index]
        X_valid = X_train_full.iloc[valid_index]

        y_train = y_train_full.iloc[train_index]

        extratrees_model = make_extratrees_model(RANDOM_STATE)
        catboost_model = make_catboost_model(RANDOM_STATE)

        extratrees_model.fit(X_train, y_train)
        catboost_model.fit(X_train, y_train)

        extratrees_valid_predictions = extratrees_model.predict(X_valid)
        catboost_valid_predictions = catboost_model.predict(X_valid)

        valid_predictions = extratrees_valid_predictions.copy()
        valid_predictions[:, 2] = catboost_valid_predictions[:, 2]

        valid_predictions[:, 2] = np.clip(
            valid_predictions[:, 2],
            SI_LOWER_BOUND,
            SI_UPPER_BOUND_Q0985,
        )

        oof_predictions[valid_index] = valid_predictions

    final_extratrees_model = make_extratrees_model(RANDOM_STATE)
    final_catboost_model = make_catboost_model(RANDOM_STATE)

    final_extratrees_model.fit(X_train_full, y_train_full)
    final_catboost_model.fit(X_train_full, y_train_full)

    extratrees_test_predictions = final_extratrees_model.predict(X_test_full)
    catboost_test_predictions = final_catboost_model.predict(X_test_full)

    test_predictions = extratrees_test_predictions.copy()
    test_predictions[:, 2] = catboost_test_predictions[:, 2]

    test_predictions[:, 2] = np.clip(
        test_predictions[:, 2],
        SI_LOWER_BOUND,
        SI_UPPER_BOUND_Q0985,
    )

    return oof_predictions, test_predictions

### 3.3 Target-wise blend `original + corr_0990`

На этом шаге строятся две версии Direct-blend ветки: на исходном наборе признаков и на `corr_0990`.

Затем они смешиваются с разными весами по таргетам:

- `IC50`: вес `corr_0990` = `0.55`;
- `CC50`: вес `corr_0990` = `0.70`;
- `SI`: вес `corr_0990` = `0.55`.

In [14]:
original_feature_columns = feature_sets["original"]
corr_0990_feature_columns = feature_sets["corr_0990"]

X_original = train_data[original_feature_columns].copy()
X_test_original = test_data[original_feature_columns].copy()

X_corr_0990 = train_data[corr_0990_feature_columns].copy()
X_test_corr_0990 = test_data[corr_0990_feature_columns].copy()

original_feature_groups = get_feature_hashes(X_original)
corr_0990_feature_groups = get_feature_hashes(X_corr_0990)

print("Обучается Direct-blend ветка на original features")
original_oof_predictions, original_test_predictions = fit_direct_blend_feature_set(
    X_train_full=X_original,
    X_test_full=X_test_original,
    y_train_full=y,
    cv=group_cv,
    groups=original_feature_groups,
)

print("Обучается Direct-blend ветка на corr_0990 features")
corr_0990_oof_predictions, corr_0990_test_predictions = (
    fit_direct_blend_feature_set(
        X_train_full=X_corr_0990,
        X_test_full=X_test_corr_0990,
        y_train_full=y,
        cv=group_cv,
        groups=corr_0990_feature_groups,
    )
)

IC50_CORR_WEIGHT = 0.55
CC50_CORR_WEIGHT = 0.70
SI_CORR_WEIGHT = 0.55

targetwise_blend_oof_predictions = np.zeros_like(corr_0990_oof_predictions)

targetwise_blend_oof_predictions[:, 0] = (
    (1 - IC50_CORR_WEIGHT) * original_oof_predictions[:, 0]
    + IC50_CORR_WEIGHT * corr_0990_oof_predictions[:, 0]
)

targetwise_blend_oof_predictions[:, 1] = (
    (1 - CC50_CORR_WEIGHT) * original_oof_predictions[:, 1]
    + CC50_CORR_WEIGHT * corr_0990_oof_predictions[:, 1]
)

targetwise_blend_oof_predictions[:, 2] = (
    (1 - SI_CORR_WEIGHT) * original_oof_predictions[:, 2]
    + SI_CORR_WEIGHT * corr_0990_oof_predictions[:, 2]
)

targetwise_blend_oof_predictions[:, 2] = np.clip(
    targetwise_blend_oof_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

targetwise_blend_predictions = np.zeros_like(corr_0990_test_predictions)

targetwise_blend_predictions[:, 0] = (
    (1 - IC50_CORR_WEIGHT) * original_test_predictions[:, 0]
    + IC50_CORR_WEIGHT * corr_0990_test_predictions[:, 0]
)

targetwise_blend_predictions[:, 1] = (
    (1 - CC50_CORR_WEIGHT) * original_test_predictions[:, 1]
    + CC50_CORR_WEIGHT * corr_0990_test_predictions[:, 1]
)

targetwise_blend_predictions[:, 2] = (
    (1 - SI_CORR_WEIGHT) * original_test_predictions[:, 2]
    + SI_CORR_WEIGHT * corr_0990_test_predictions[:, 2]
)

targetwise_blend_predictions[:, 2] = np.clip(
    targetwise_blend_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

targetwise_blend_score = competition_rmse(y, targetwise_blend_oof_predictions)
targetwise_blend_score_by_target = competition_rmse_by_target(
    y,
    targetwise_blend_oof_predictions,
    TARGET_COLUMNS,
)

print(f"OOF RMSE target-wise blend: {targetwise_blend_score:.6f}")
print("OOF RMSE по таргетам:")
print(targetwise_blend_score_by_target)

describe_predictions(targetwise_blend_predictions, TARGET_COLUMNS)

Обучается Direct-blend ветка на original features
Обучается fold 1
Обучается fold 2
Обучается fold 3
Обучается fold 4
Обучается fold 5
Обучается Direct-blend ветка на corr_0990 features
Обучается fold 1
Обучается fold 2
Обучается fold 3
Обучается fold 4
Обучается fold 5
OOF RMSE target-wise blend: 516.000070
OOF RMSE по таргетам:
IC50   343.653599
CC50   449.658819
SI     754.687792
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,249.452416,337.361196,0.341586,59.734199,134.827315,292.761317,2309.597757
CC50,250.000000,656.956284,565.638627,30.064449,279.982460,505.371008,909.542020,3314.114044
SI,250.000000,48.084596,97.021278,1.000000,7.949150,21.137523,40.003032,635.678571


### 3.4 Mean-aggregation pipeline

В train есть одинаковые feature-векторы с разными таргетами. Для дополнительной ветки одинаковые feature-векторы агрегируются внутри train через среднее значение таргетов.

Эта ветка не заменяет Direct-blend, а используется как дополнительный pipeline для последующего meta-blend.

In [15]:
def aggregate_training_data(X_train, y_train, method):
    """Агрегирует одинаковые feature-векторы в train."""
    feature_hashes = get_feature_hashes(X_train)

    feature_hash_table = pd.DataFrame(
        {
            "feature_hash": feature_hashes.to_numpy(),
        }
    )

    train_table = pd.concat(
        [
            X_train.reset_index(drop=True),
            y_train.reset_index(drop=True),
            feature_hash_table,
        ],
        axis=1,
    ).copy()

    current_feature_columns = X_train.columns.tolist()
    grouped = train_table.groupby("feature_hash", sort=False)

    X_aggregated = grouped[current_feature_columns].first()

    if method == "mean":
        y_aggregated = grouped[TARGET_COLUMNS].mean()
    else:
        raise ValueError("В final pipeline используется только method='mean'.")

    group_sizes = grouped.size()

    return (
        X_aggregated.reset_index(drop=True),
        y_aggregated.reset_index(drop=True),
        group_sizes.reset_index(drop=True),
    )


def fit_direct_blend_feature_set_with_aggregation(
    X_train,
    y_train,
    X_valid,
    aggregation_method,
):
    """Обучает ExtraTrees и CatBoost на агрегированном train."""
    X_aggregated, y_aggregated, group_sizes = aggregate_training_data(
        X_train=X_train,
        y_train=y_train,
        method=aggregation_method,
    )

    extratrees_model = make_extratrees_model(RANDOM_STATE)
    catboost_model = make_catboost_model(RANDOM_STATE)

    extratrees_model.fit(X_aggregated, y_aggregated)
    catboost_model.fit(X_aggregated, y_aggregated)

    extratrees_predictions = extratrees_model.predict(X_valid)
    catboost_predictions = catboost_model.predict(X_valid)

    predictions = extratrees_predictions.copy()
    predictions[:, 2] = catboost_predictions[:, 2]

    return predictions


def fit_full_direct_blend_feature_set_with_aggregation(
    X_train,
    y_train,
    X_test,
    aggregation_method,
):
    """Обучает ветку на агрегированном полном train и делает test-прогноз."""
    X_aggregated, y_aggregated, group_sizes = aggregate_training_data(
        X_train=X_train,
        y_train=y_train,
        method=aggregation_method,
    )

    extratrees_model = make_extratrees_model(RANDOM_STATE)
    catboost_model = make_catboost_model(RANDOM_STATE)

    extratrees_model.fit(X_aggregated, y_aggregated)
    catboost_model.fit(X_aggregated, y_aggregated)

    extratrees_predictions = extratrees_model.predict(X_test)
    catboost_predictions = catboost_model.predict(X_test)

    predictions = extratrees_predictions.copy()
    predictions[:, 2] = catboost_predictions[:, 2]

    print(f"Размер train после агрегации: {len(X_aggregated)}")
    print(f"Средний размер группы дублей: {group_sizes.mean():.3f}")

    return predictions

In [16]:
AGGREGATION_METHOD = "mean"

mean_aggregation_oof_predictions = np.zeros_like(targetwise_blend_oof_predictions)

for fold_number, (train_index, valid_index) in enumerate(
    group_cv.split(X_original, y, original_feature_groups),
    start=1,
):
    print(f"Обучается aggregation fold {fold_number}")

    y_train = y.iloc[train_index]

    original_fold_predictions = fit_direct_blend_feature_set_with_aggregation(
        X_train=X_original.iloc[train_index],
        y_train=y_train,
        X_valid=X_original.iloc[valid_index],
        aggregation_method=AGGREGATION_METHOD,
    )

    corr_fold_predictions = fit_direct_blend_feature_set_with_aggregation(
        X_train=X_corr_0990.iloc[train_index],
        y_train=y_train,
        X_valid=X_corr_0990.iloc[valid_index],
        aggregation_method=AGGREGATION_METHOD,
    )

    fold_predictions = np.zeros_like(corr_fold_predictions)

    fold_predictions[:, 0] = (
        (1 - IC50_CORR_WEIGHT) * original_fold_predictions[:, 0]
        + IC50_CORR_WEIGHT * corr_fold_predictions[:, 0]
    )

    fold_predictions[:, 1] = (
        (1 - CC50_CORR_WEIGHT) * original_fold_predictions[:, 1]
        + CC50_CORR_WEIGHT * corr_fold_predictions[:, 1]
    )

    fold_predictions[:, 2] = (
        (1 - SI_CORR_WEIGHT) * original_fold_predictions[:, 2]
        + SI_CORR_WEIGHT * corr_fold_predictions[:, 2]
    )

    fold_predictions[:, 2] = np.clip(
        fold_predictions[:, 2],
        SI_LOWER_BOUND,
        SI_UPPER_BOUND_Q0985,
    )

    mean_aggregation_oof_predictions[valid_index] = fold_predictions

original_aggregated_test_predictions = (
    fit_full_direct_blend_feature_set_with_aggregation(
        X_train=X_original,
        y_train=y,
        X_test=X_test_original,
        aggregation_method=AGGREGATION_METHOD,
    )
)

corr_0990_aggregated_test_predictions = (
    fit_full_direct_blend_feature_set_with_aggregation(
        X_train=X_corr_0990,
        y_train=y,
        X_test=X_test_corr_0990,
        aggregation_method=AGGREGATION_METHOD,
    )
)

aggregated_blend_predictions = np.zeros_like(corr_0990_aggregated_test_predictions)

aggregated_blend_predictions[:, 0] = (
    (1 - IC50_CORR_WEIGHT) * original_aggregated_test_predictions[:, 0]
    + IC50_CORR_WEIGHT * corr_0990_aggregated_test_predictions[:, 0]
)

aggregated_blend_predictions[:, 1] = (
    (1 - CC50_CORR_WEIGHT) * original_aggregated_test_predictions[:, 1]
    + CC50_CORR_WEIGHT * corr_0990_aggregated_test_predictions[:, 1]
)

aggregated_blend_predictions[:, 2] = (
    (1 - SI_CORR_WEIGHT) * original_aggregated_test_predictions[:, 2]
    + SI_CORR_WEIGHT * corr_0990_aggregated_test_predictions[:, 2]
)

aggregated_blend_predictions[:, 2] = np.clip(
    aggregated_blend_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

mean_aggregation_score = competition_rmse(y, mean_aggregation_oof_predictions)

print(f"OOF RMSE mean-aggregation pipeline: {mean_aggregation_score:.6f}")
describe_predictions(aggregated_blend_predictions, TARGET_COLUMNS)

Обучается aggregation fold 1
Обучается aggregation fold 2
Обучается aggregation fold 3
Обучается aggregation fold 4
Обучается aggregation fold 5
Размер train после агрегации: 630
Средний размер группы дублей: 1.192
Размер train после агрегации: 630
Средний размер группы дублей: 1.192
OOF RMSE mean-aggregation pipeline: 523.497235


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,248.658473,327.446892,4.611389,73.169150,139.212449,273.966051,2261.799868
CC50,250.000000,649.183086,551.252119,28.494120,277.970845,499.026423,896.811753,3251.542111
SI,250.000000,48.269716,99.891914,1.000000,5.551181,18.629074,43.860393,635.678571


### 3.5 KNN-сигнал для `SI`

К mean-aggregation pipeline добавляется KNN-сигнал только для `SI`.

Финальная конфигурация:

- feature set: `original`;
- scaler: `StandardScaler`;
- `n_neighbors=3`;
- `weights="distance"`;
- вес KNN для `SI`: `0.30`.

In [17]:
def fit_knn_si_oof_and_test(
    X_train_full,
    X_test_full,
    y_train_full,
    cv,
    groups,
    n_neighbors,
    knn_weights,
):
    """Строит OOF и test-прогноз SI через KNN на mean-агрегированном train."""
    oof_si_predictions = np.zeros(len(y_train_full))

    for fold_number, (train_index, valid_index) in enumerate(
        cv.split(X_train_full, y_train_full, groups),
        start=1,
    ):
        print(f"Обучается KNN fold {fold_number}")

        X_train = X_train_full.iloc[train_index]
        X_valid = X_train_full.iloc[valid_index]
        y_train = y_train_full.iloc[train_index]

        X_aggregated, y_aggregated, group_sizes = aggregate_training_data(
            X_train=X_train,
            y_train=y_train,
            method="mean",
        )

        knn_model = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    KNeighborsRegressor(
                        n_neighbors=n_neighbors,
                        weights=knn_weights,
                        metric="minkowski",
                        p=2,
                    ),
                ),
            ]
        )

        knn_model.fit(X_aggregated, y_aggregated["SI"])
        oof_si_predictions[valid_index] = knn_model.predict(X_valid)

    X_aggregated, y_aggregated, group_sizes = aggregate_training_data(
        X_train=X_train_full,
        y_train=y_train_full,
        method="mean",
    )

    final_knn_model = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                KNeighborsRegressor(
                    n_neighbors=n_neighbors,
                    weights=knn_weights,
                    metric="minkowski",
                    p=2,
                ),
            ),
        ]
    )

    final_knn_model.fit(X_aggregated, y_aggregated["SI"])
    test_si_predictions = final_knn_model.predict(X_test_full)

    print(f"Размер train после агрегации для KNN: {len(X_aggregated)}")
    print(f"Средний размер группы дублей: {group_sizes.mean():.3f}")

    return oof_si_predictions, test_si_predictions


KNN_SI_WEIGHT = 0.30

knn_si_oof_predictions, knn_si_test_predictions = fit_knn_si_oof_and_test(
    X_train_full=X_original,
    X_test_full=X_test_original,
    y_train_full=y,
    cv=group_cv,
    groups=original_feature_groups,
    n_neighbors=3,
    knn_weights="distance",
)

aggregation_knn_oof = mean_aggregation_oof_predictions.copy()

aggregation_knn_oof[:, 2] = (
    (1 - KNN_SI_WEIGHT) * mean_aggregation_oof_predictions[:, 2]
    + KNN_SI_WEIGHT * knn_si_oof_predictions
)

aggregation_knn_oof[:, 2] = np.clip(
    aggregation_knn_oof[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

aggregation_knn_predictions = aggregated_blend_predictions.copy()

aggregation_knn_predictions[:, 2] = (
    (1 - KNN_SI_WEIGHT) * aggregated_blend_predictions[:, 2]
    + KNN_SI_WEIGHT * knn_si_test_predictions
)

aggregation_knn_predictions[:, 2] = np.clip(
    aggregation_knn_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

aggregation_knn_score = competition_rmse(y, aggregation_knn_oof)

print(f"OOF RMSE aggregation + KNN: {aggregation_knn_score:.6f}")
describe_predictions(aggregation_knn_predictions, TARGET_COLUMNS)

Обучается KNN fold 1
Обучается KNN fold 2
Обучается KNN fold 3
Обучается KNN fold 4
Обучается KNN fold 5
Размер train после агрегации для KNN: 630
Средний размер группы дублей: 1.192
OOF RMSE aggregation + KNN: 521.947319


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,248.658473,327.446892,4.611389,73.169150,139.212449,273.966051,2261.799868
CC50,250.000000,649.183086,551.252119,28.494120,277.970845,499.026423,896.811753,3251.542111
SI,250.000000,46.985968,102.676355,1.000000,6.717154,16.937488,37.069974,635.678571


### 3.6 Fine meta target-wise blend

На этом шаге смешиваются две ветки:

- non-aggregated target-wise blend;
- mean-aggregation pipeline с KNN-сигналом для `SI`.

Финальные веса aggregation/KNN pipeline:

- `IC50`: `0.15`;
- `CC50`: `0.00`;
- `SI`: `0.65`.

In [18]:
IC50_AGGREGATION_WEIGHT = 0.15
CC50_AGGREGATION_WEIGHT = 0.00
SI_AGGREGATION_WEIGHT = 0.65

fine_meta_oof_predictions = np.zeros_like(aggregation_knn_oof)

fine_meta_oof_predictions[:, 0] = (
    (1 - IC50_AGGREGATION_WEIGHT) * targetwise_blend_oof_predictions[:, 0]
    + IC50_AGGREGATION_WEIGHT * aggregation_knn_oof[:, 0]
)

fine_meta_oof_predictions[:, 1] = (
    (1 - CC50_AGGREGATION_WEIGHT) * targetwise_blend_oof_predictions[:, 1]
    + CC50_AGGREGATION_WEIGHT * aggregation_knn_oof[:, 1]
)

fine_meta_oof_predictions[:, 2] = (
    (1 - SI_AGGREGATION_WEIGHT) * targetwise_blend_oof_predictions[:, 2]
    + SI_AGGREGATION_WEIGHT * aggregation_knn_oof[:, 2]
)

fine_meta_oof_predictions[:, 2] = np.clip(
    fine_meta_oof_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

fine_meta_targetwise_predictions = np.zeros_like(aggregation_knn_predictions)

fine_meta_targetwise_predictions[:, 0] = (
    (1 - IC50_AGGREGATION_WEIGHT) * targetwise_blend_predictions[:, 0]
    + IC50_AGGREGATION_WEIGHT * aggregation_knn_predictions[:, 0]
)

fine_meta_targetwise_predictions[:, 1] = (
    (1 - CC50_AGGREGATION_WEIGHT) * targetwise_blend_predictions[:, 1]
    + CC50_AGGREGATION_WEIGHT * aggregation_knn_predictions[:, 1]
)

fine_meta_targetwise_predictions[:, 2] = (
    (1 - SI_AGGREGATION_WEIGHT) * targetwise_blend_predictions[:, 2]
    + SI_AGGREGATION_WEIGHT * aggregation_knn_predictions[:, 2]
)

fine_meta_targetwise_predictions[:, 2] = np.clip(
    fine_meta_targetwise_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

fine_meta_score = competition_rmse(y, fine_meta_oof_predictions)
fine_meta_score_by_target = competition_rmse_by_target(
    y,
    fine_meta_oof_predictions,
    TARGET_COLUMNS,
)

print(f"OOF RMSE fine meta target-wise blend: {fine_meta_score:.6f}")
print("OOF RMSE по таргетам:")
print(fine_meta_score_by_target)

describe_predictions(fine_meta_targetwise_predictions, TARGET_COLUMNS)

OOF RMSE fine meta target-wise blend: 515.848567
OOF RMSE по таргетам:
IC50   343.550136
CC50   449.658819
SI     754.336744
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,249.333325,335.392010,4.110939,66.180049,134.719575,290.081545,2302.428073
CC50,250.000000,656.956284,565.638627,30.064449,279.982460,505.371008,909.542020,3314.114044
SI,250.000000,47.370488,99.964692,1.000000,6.862403,18.806363,38.399287,635.678571


In [19]:
current_best_oof = fine_meta_oof_predictions.copy()
current_best_test_predictions = fine_meta_targetwise_predictions.copy()

current_best_score = competition_rmse(y, current_best_oof)
current_best_score_by_target = competition_rmse_by_target(
    y,
    current_best_oof,
    TARGET_COLUMNS,
)

print(f"OOF RMSE current hybrid-best before calibration: {current_best_score:.6f}")
print("OOF RMSE по таргетам:")
print(current_best_score_by_target)

OOF RMSE current hybrid-best before calibration: 515.848567
OOF RMSE по таргетам:
IC50   343.550136
CC50   449.658819
SI     754.336744
dtype: float64


### 3.7 Ratio-ветка и isotonic-калибровка

Для `IC50` и `CC50` дополнительно используется ratio-ветка. Она обучает CatBoost-модель на `log1p(IC50)` и `log1p(CC50)`, а `SI` рассчитывает как `CC50 / IC50`.

Как самостоятельное решение эта ветка была слабее, но после isotonic-калибровки дала полезный дополнительный сигнал для `IC50` и `CC50`.

In [20]:
calculated_si_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)


def train_calculated_si_branch(
    X_train_full,
    X_test_full,
    y_train_full,
    cv,
    stratification_target,
    feature_columns,
):
    """Обучает ratio-ветку: CatBoost для IC50/CC50 и SI = CC50 / IC50."""
    target_columns_for_model = ["IC50", "CC50"]

    X_selected = X_train_full[feature_columns].copy()
    X_test_selected = X_test_full[feature_columns].copy()

    y_log = np.log1p(y_train_full[target_columns_for_model])

    oof_predictions = np.zeros((len(X_selected), len(TARGET_COLUMNS)))
    test_predictions = np.zeros((len(X_test_selected), 2))

    for fold_number, (train_index, valid_index) in enumerate(
        cv.split(X_selected, stratification_target),
        start=1,
    ):
        print(f"Обучается ratio fold {fold_number}")

        X_train = X_selected.iloc[train_index]
        X_valid = X_selected.iloc[valid_index]

        y_train_log = y_log.iloc[train_index]
        y_valid = y_train_full.iloc[valid_index]

        train_pool = Pool(X_train, y_train_log)
        valid_pool = Pool(
            X_valid,
            np.log1p(y_valid[target_columns_for_model]),
        )

        model = CatBoostRegressor(
            iterations=2500,
            learning_rate=0.02,
            depth=6,
            l2_leaf_reg=10,
            loss_function="MultiRMSE",
            random_seed=RANDOM_STATE + fold_number - 1,
            verbose=False,
            allow_writing_files=False,
        )

        model.fit(
            train_pool,
            eval_set=valid_pool,
            early_stopping_rounds=200,
        )

        valid_predictions_log = model.predict(X_valid)
        valid_base_predictions = np.expm1(valid_predictions_log)

        valid_ic50 = valid_base_predictions[:, 0]
        valid_cc50 = valid_base_predictions[:, 1]

        safe_valid_ic50 = np.maximum(valid_ic50, 1e-9)
        valid_si = valid_cc50 / safe_valid_ic50

        valid_si = np.clip(
            valid_si,
            y_train_full.iloc[train_index]["SI"].min(),
            y_train_full.iloc[train_index]["SI"].max(),
        )

        valid_predictions = np.column_stack(
            [
                valid_ic50,
                valid_cc50,
                valid_si,
            ]
        )

        oof_predictions[valid_index] = valid_predictions

        test_predictions_log = model.predict(X_test_selected)
        test_predictions += np.expm1(test_predictions_log) / cv.n_splits

    test_ic50 = test_predictions[:, 0]
    test_cc50 = test_predictions[:, 1]

    safe_test_ic50 = np.maximum(test_ic50, 1e-9)
    test_si = test_cc50 / safe_test_ic50

    test_si = np.clip(
        test_si,
        y_train_full["SI"].min(),
        y_train_full["SI"].max(),
    )

    full_test_predictions = np.column_stack(
        [
            test_ic50,
            test_cc50,
            test_si,
        ]
    )

    return oof_predictions, full_test_predictions


best_calculated_si_oof, best_calculated_si_test_predictions = (
    train_calculated_si_branch(
        X_train_full=train_data,
        X_test_full=test_data,
        y_train_full=y,
        cv=calculated_si_cv,
        stratification_target=si_stratification_target,
        feature_columns=feature_sets["corr_0990"],
    )
)

calculated_si_score = competition_rmse(y, best_calculated_si_oof)

print(f"OOF RMSE ratio-ветки: {calculated_si_score:.6f}")
describe_predictions(best_calculated_si_test_predictions, TARGET_COLUMNS)

Обучается ratio fold 1
Обучается ratio fold 2
Обучается ratio fold 3
Обучается ratio fold 4
Обучается ratio fold 5
OOF RMSE ratio-ветки: 551.882420


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,119.479302,180.168302,0.520449,25.725951,56.447898,120.943637,1497.443872
CC50,250.000000,450.122381,422.498686,14.623654,169.240900,309.921774,590.779138,2833.556933
SI,250.000000,8.781101,14.852702,1.317965,3.048904,4.692379,8.827483,117.117119


In [21]:
def apply_isotonic_calibration(
    train_predictions,
    train_target,
    test_predictions,
):
    """Применяет монотонную isotonic-калибровку."""
    calibration_model = IsotonicRegression(
        out_of_bounds="clip",
    )

    calibration_model.fit(
        train_predictions,
        train_target,
    )

    calibrated_train_predictions = calibration_model.predict(
        train_predictions
    )

    calibrated_test_predictions = calibration_model.predict(
        test_predictions
    )

    return calibrated_train_predictions, calibrated_test_predictions


def clip_to_train_target_range(predictions, target_values):
    """Ограничивает предсказания диапазоном train-таргета."""
    return np.clip(
        predictions,
        target_values.min(),
        target_values.max(),
    )


isotonic_branch_oof = best_calculated_si_oof.copy()
isotonic_branch_test = best_calculated_si_test_predictions.copy()

for target_index, target_column in enumerate(["IC50", "CC50"]):
    calibrated_train_target, calibrated_test_target = (
        apply_isotonic_calibration(
            train_predictions=best_calculated_si_oof[:, target_index],
            train_target=y[target_column].values,
            test_predictions=best_calculated_si_test_predictions[:, target_index],
        )
    )

    isotonic_branch_oof[:, target_index] = clip_to_train_target_range(
        calibrated_train_target,
        y[target_column],
    )

    isotonic_branch_test[:, target_index] = clip_to_train_target_range(
        calibrated_test_target,
        y[target_column],
    )

safe_oof_ic50 = np.maximum(isotonic_branch_oof[:, 0], 1e-9)
safe_test_ic50 = np.maximum(isotonic_branch_test[:, 0], 1e-9)

isotonic_branch_oof[:, 2] = isotonic_branch_oof[:, 1] / safe_oof_ic50
isotonic_branch_test[:, 2] = isotonic_branch_test[:, 1] / safe_test_ic50

isotonic_branch_oof[:, 2] = np.clip(
    isotonic_branch_oof[:, 2],
    y["SI"].min(),
    y["SI"].max(),
)

isotonic_branch_test[:, 2] = np.clip(
    isotonic_branch_test[:, 2],
    y["SI"].min(),
    y["SI"].max(),
)

isotonic_branch_score = competition_rmse(y, isotonic_branch_oof)

print(f"OOF RMSE isotonic ratio-ветки: {isotonic_branch_score:.6f}")
describe_predictions(isotonic_branch_test, TARGET_COLUMNS)

OOF RMSE isotonic ratio-ветки: 476.325257


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,229.367828,216.640590,0.088549,61.396364,131.071824,325.467408,1309.128993
CC50,250.000000,632.987477,472.293374,21.771460,276.827042,492.711969,859.233257,3597.992382
SI,250.000000,26.584615,181.588492,1.006301,1.882782,3.057390,5.563809,1661.507588


### 3.8 Финальный calibrated hybrid-best

Финальный calibrated hybrid-best добавляет isotonic-калиброванную ratio-ветку только к `IC50` и `CC50`.

Финальные веса:

- `IC50`: `0.075` isotonic branch;
- `CC50`: `0.40` isotonic branch;
- `SI`: `0.00` isotonic branch.

`SI` остаётся из fine meta target-wise blend.

In [22]:
CALIBRATED_IC50_WEIGHT = 0.075
CALIBRATED_CC50_WEIGHT = 0.40
CALIBRATED_SI_WEIGHT = 0.00

calibrated_hybrid_best_oof = current_best_oof.copy()

calibrated_hybrid_best_oof[:, 0] = (
    (1 - CALIBRATED_IC50_WEIGHT) * current_best_oof[:, 0]
    + CALIBRATED_IC50_WEIGHT * isotonic_branch_oof[:, 0]
)

calibrated_hybrid_best_oof[:, 1] = (
    (1 - CALIBRATED_CC50_WEIGHT) * current_best_oof[:, 1]
    + CALIBRATED_CC50_WEIGHT * isotonic_branch_oof[:, 1]
)

calibrated_hybrid_best_oof[:, 2] = current_best_oof[:, 2]

calibrated_hybrid_best_oof[:, 2] = np.clip(
    calibrated_hybrid_best_oof[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

calibrated_hybrid_best_test_predictions = current_best_test_predictions.copy()

calibrated_hybrid_best_test_predictions[:, 0] = (
    (1 - CALIBRATED_IC50_WEIGHT) * current_best_test_predictions[:, 0]
    + CALIBRATED_IC50_WEIGHT * isotonic_branch_test[:, 0]
)

calibrated_hybrid_best_test_predictions[:, 1] = (
    (1 - CALIBRATED_CC50_WEIGHT) * current_best_test_predictions[:, 1]
    + CALIBRATED_CC50_WEIGHT * isotonic_branch_test[:, 1]
)

calibrated_hybrid_best_test_predictions[:, 2] = current_best_test_predictions[:, 2]

calibrated_hybrid_best_test_predictions[:, 2] = np.clip(
    calibrated_hybrid_best_test_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

calibrated_hybrid_best_score = competition_rmse(
    y,
    calibrated_hybrid_best_oof,
)

calibrated_hybrid_best_score_by_target = competition_rmse_by_target(
    y,
    calibrated_hybrid_best_oof,
    TARGET_COLUMNS,
)

print(f"OOF RMSE calibrated hybrid-best: {calibrated_hybrid_best_score:.6f}")
print("OOF RMSE по таргетам:")
print(calibrated_hybrid_best_score_by_target)

describe_predictions(calibrated_hybrid_best_test_predictions, TARGET_COLUMNS)

OOF RMSE calibrated hybrid-best: 510.649345
OOF RMSE по таргетам:
IC50   339.228295
CC50   438.382996
SI     754.336744
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,247.835913,323.867007,3.809260,67.463192,136.319914,290.872287,2227.930642
CC50,250.000000,647.368761,514.317197,26.896212,310.398642,524.567703,845.284029,3427.665379
SI,250.000000,47.370488,99.964692,1.000000,6.862403,18.806363,38.399287,635.678571


## 4. Вторая ветка: CatBoost MAE

В этом разделе воспроизводится вторая сильная ветка решения — CatBoost MAE.

Ветка строится независимо от calibrated hybrid-best:

- пропуски заполняются медианами train;
- удаляются константные признаки;
- через CatBoost importance отбираются top-признаки по каждому таргету;
- для каждого таргета обучается отдельная CatBoost MAE модель;
- предсказания стабилизируются через seed averaging;
- для `SI` применяется postprocessing: смешивание direct-прогноза и formula-SI.

### 4.1 Подготовка признаков с заполнением пропусков

In [23]:
mae_train_data = train_data.copy()
mae_test_data = test_data.copy()

numeric_train_medians = mae_train_data.median(numeric_only=True)

mae_train_data = mae_train_data.fillna(numeric_train_medians)
mae_test_data = mae_test_data.fillna(numeric_train_medians)

mae_constant_columns = [
    column
    for column in mae_train_data.columns
    if mae_train_data[column].nunique() <= 1
]

mae_feature_columns = mae_train_data.drop(
    columns=[ID_COLUMN, *TARGET_COLUMNS, *mae_constant_columns],
    errors="ignore",
).columns.tolist()

mae_train_features = mae_train_data[mae_feature_columns].copy()
mae_test_features = mae_test_data[mae_feature_columns].copy()

print(f"Количество признаков до отбора: {len(mae_feature_columns)}")
print(f"Количество константных признаков: {len(mae_constant_columns)}")
print(f"Пропуски в train-признаках: {mae_train_features.isna().sum().sum()}")
print(f"Пропуски в test-признаках: {mae_test_features.isna().sum().sum()}")

Количество признаков до отбора: 192
Количество константных признаков: 18
Пропуски в train-признаках: 0
Пропуски в test-признаках: 0


### 4.2 Отбор признаков через CatBoost importance

Для каждого таргета обучается отдельная CatBoost-модель, после чего отбираются top-30 признаков по важности. Затем списки признаков объединяются в один набор для CatBoost MAE ветки.

In [24]:
MAE_TOP_FEATURES_PER_TARGET = 30

mae_important_feature_candidates = []
mae_feature_importance_tables = {}

for target_column in TARGET_COLUMNS:
    print(f"Отбор признаков для таргета: {target_column}")

    feature_selection_model = CatBoostRegressor(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="MAE",
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )

    feature_selection_model.fit(
        mae_train_features,
        mae_train_data[target_column],
    )

    feature_importance = pd.DataFrame(
        {
            "feature": mae_train_features.columns,
            "importance": feature_selection_model.get_feature_importance(),
        }
    ).sort_values("importance", ascending=False)

    mae_feature_importance_tables[target_column] = feature_importance

    top_features = (
        feature_importance
        .head(MAE_TOP_FEATURES_PER_TARGET)["feature"]
        .tolist()
    )

    mae_important_feature_candidates.extend(top_features)

mae_selected_feature_columns = list(
    dict.fromkeys(mae_important_feature_candidates)
)

print(f"Отобрано признаков: {len(mae_selected_feature_columns)}")

mae_selected_features_df = pd.DataFrame(
    {"feature": mae_selected_feature_columns}
)

display(mae_selected_features_df.head(30))

Отбор признаков для таргета: IC50
Отбор признаков для таргета: CC50
Отбор признаков для таргета: SI
Отобрано признаков: 68


,feature
0,BCUT2D_MRLOW
1,MinAbsEStateIndex
2,NumSaturatedHeterocycles
3,NumHDonors
4,LabuteASA
5,EState_VSA8
6,Chi2n
7,PEOE_VSA7
8,VSA_EState8
9,Chi0v


### 4.3 Функция обучения CatBoost MAE с несколькими seed

In [25]:
def train_catboost_mae_seed_ensemble(
    train_features,
    test_features,
    y,
    feature_columns,
    cv,
    stratification_target,
    n_seeds,
):
    """Обучает CatBoost MAE отдельно по таргетам и усредняет по seed."""
    selected_train_features = train_features[feature_columns].copy()
    selected_test_features = test_features[feature_columns].copy()

    oof_direct_predictions = np.zeros((len(selected_train_features), 3))
    test_direct_predictions = np.zeros((len(selected_test_features), 3))

    fold_rows = []

    for seed_number in range(n_seeds):
        print(f"Seed {seed_number + 1} из {n_seeds}")

        for target_index, target_column in enumerate(TARGET_COLUMNS):
            target_values = y[target_column].values

            for fold_number, (train_index, valid_index) in enumerate(
                cv.split(selected_train_features, stratification_target),
                start=1,
            ):
                print(f"{target_column}: fold {fold_number}")

                X_train = selected_train_features.iloc[train_index]
                X_valid = selected_train_features.iloc[valid_index]

                y_train = target_values[train_index]
                y_valid = target_values[valid_index]

                model = CatBoostRegressor(
                    iterations=1500,
                    learning_rate=0.03,
                    depth=6,
                    l2_leaf_reg=7,
                    loss_function="MAE",
                    eval_metric="RMSE",
                    random_seed=seed_number * 100 + fold_number - 1,
                    verbose=False,
                    allow_writing_files=False,
                )

                model.fit(
                    X_train,
                    y_train,
                    eval_set=(X_valid, y_valid),
                    early_stopping_rounds=150,
                )

                valid_predictions = model.predict(X_valid)
                test_predictions = model.predict(selected_test_features)

                oof_direct_predictions[valid_index, target_index] += (
                    valid_predictions / n_seeds
                )

                test_direct_predictions[:, target_index] += (
                    test_predictions / (cv.n_splits * n_seeds)
                )

                fold_rmse = np.sqrt(
                    mean_squared_error(
                        y_valid,
                        valid_predictions,
                    )
                )

                fold_rows.append(
                    {
                        "target": target_column,
                        "seed": seed_number,
                        "fold": fold_number,
                        "best_iteration": model.get_best_iteration(),
                        "rmse": fold_rmse,
                    }
                )

    fold_results = pd.DataFrame(fold_rows)

    return oof_direct_predictions, test_direct_predictions, fold_results

### 4.4 Обучение CatBoost MAE seed averaging

In [26]:
MAE_N_SEEDS = 5

mae_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

mae_stratification_target = (y["SI"] > 8).astype(int)

mae_oof_direct_predictions, mae_test_direct_predictions, (
    mae_fold_results
) = train_catboost_mae_seed_ensemble(
    train_features=mae_train_features,
    test_features=mae_test_features,
    y=y,
    feature_columns=mae_selected_feature_columns,
    cv=mae_cv,
    stratification_target=mae_stratification_target,
    n_seeds=MAE_N_SEEDS,
)

display(mae_fold_results.head(20))
display(
    mae_fold_results
    .groupby("target", as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_best_iteration=("best_iteration", "mean"),
    )
)

Seed 1 из 5
IC50: fold 1
IC50: fold 2
IC50: fold 3
IC50: fold 4
IC50: fold 5
CC50: fold 1
CC50: fold 2
CC50: fold 3
CC50: fold 4
CC50: fold 5
SI: fold 1
SI: fold 2
SI: fold 3
SI: fold 4
SI: fold 5
Seed 2 из 5
IC50: fold 1
IC50: fold 2
IC50: fold 3
IC50: fold 4
IC50: fold 5
CC50: fold 1
CC50: fold 2
CC50: fold 3
CC50: fold 4
CC50: fold 5
SI: fold 1
SI: fold 2
SI: fold 3
SI: fold 4
SI: fold 5
Seed 3 из 5
IC50: fold 1
IC50: fold 2
IC50: fold 3
IC50: fold 4
IC50: fold 5
CC50: fold 1
CC50: fold 2
CC50: fold 3
CC50: fold 4
CC50: fold 5
SI: fold 1
SI: fold 2
SI: fold 3
SI: fold 4
SI: fold 5
Seed 4 из 5
IC50: fold 1
IC50: fold 2
IC50: fold 3
IC50: fold 4
IC50: fold 5
CC50: fold 1
CC50: fold 2
CC50: fold 3
CC50: fold 4
CC50: fold 5
SI: fold 1
SI: fold 2
SI: fold 3
SI: fold 4
SI: fold 5
Seed 5 из 5
IC50: fold 1
IC50: fold 2
IC50: fold 3
IC50: fold 4
IC50: fold 5
CC50: fold 1
CC50: fold 2
CC50: fold 3
CC50: fold 4
CC50: fold 5
SI: fold 1
SI: fold 2
SI: fold 3
SI: fold 4
SI: fold 5


,target,seed,fold,best_iteration,rmse
0,IC50,0,1,822,287.666818
1,IC50,0,2,285,412.909429
2,IC50,0,3,424,374.518445
3,IC50,0,4,287,257.201675
4,IC50,0,5,524,276.929215
5,CC50,0,1,261,520.080547
6,CC50,0,2,1175,454.688492
7,CC50,0,3,479,588.278676
8,CC50,0,4,773,396.268348
9,CC50,0,5,131,392.342048


,target,mean_rmse,std_rmse,mean_best_iteration
0,CC50,467.003793,76.301730,557.000000
1,IC50,323.748531,61.437209,550.280000
2,SI,506.087338,613.881672,670.560000


### 4.5 Постобработка предсказаний и расчёт `SI`

В postprocessing применяются четыре шага:

1. нижнее ограничение `IC50` и `CC50`;
2. расчёт formula-SI как `CC50 / IC50`;
3. смешивание direct-`SI` и formula-`SI`;
4. clipping всех таргетов по верхним квантилям train.

In [27]:
def make_mae_final_predictions(
    direct_predictions,
    y_train,
    si_formula_weight=0.5,
    clip_quantile=0.995,
):
    """Готовит финальные предсказания для CatBoost MAE ветки."""
    final_predictions = direct_predictions.copy()

    final_predictions[:, 0] = np.maximum(0.001, final_predictions[:, 0])
    final_predictions[:, 1] = np.maximum(0.001, final_predictions[:, 1])

    calculated_si = final_predictions[:, 1] / np.maximum(
        final_predictions[:, 0],
        0.001,
    )

    final_predictions[:, 2] = (
        (1 - si_formula_weight) * final_predictions[:, 2]
        + si_formula_weight * calculated_si
    )

    for target_index, target_column in enumerate(TARGET_COLUMNS):
        upper_limit = y_train[target_column].quantile(clip_quantile)

        final_predictions[:, target_index] = np.clip(
            final_predictions[:, target_index],
            0.001,
            upper_limit,
        )

    return final_predictions


mae_oof_predictions = make_mae_final_predictions(
    direct_predictions=mae_oof_direct_predictions,
    y_train=y,
    si_formula_weight=0.5,
    clip_quantile=0.995,
)

mae_test_predictions = make_mae_final_predictions(
    direct_predictions=mae_test_direct_predictions,
    y_train=y,
    si_formula_weight=0.5,
    clip_quantile=0.995,
)

mae_oof_score = competition_rmse(y, mae_oof_predictions)
mae_oof_score_by_target = competition_rmse_by_target(
    y,
    mae_oof_predictions,
    TARGET_COLUMNS,
)

print("OOF RMSE CatBoost MAE seed averaging:")
print(mae_oof_score)

display(
    pd.DataFrame(
        {
            "target": TARGET_COLUMNS,
            "rmse": mae_oof_score_by_target.values,
        }
    )
)

print("Диапазон test-предсказаний:")
display(
    pd.DataFrame(
        mae_test_predictions,
        columns=TARGET_COLUMNS,
    ).describe().T
)

OOF RMSE CatBoost MAE seed averaging:
532.6353105778248


,target,rmse
0,IC50,327.909456
1,CC50,470.236947
2,SI,799.759530


Диапазон test-предсказаний:


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,177.956468,248.337404,0.542931,39.838308,87.421584,217.343309,1488.545439
CC50,250.000000,583.551342,457.336003,22.169839,270.928795,460.232612,788.050155,2857.502223
SI,250.000000,13.749050,27.806036,1.311152,3.661830,6.120948,12.850790,241.475087


### 4.6 Проверка веса formula-SI

В CatBoost MAE ветке отдельно фиксируется вес formula-SI. В исследовательском ноутбуке лучший public score в этой ветке дал вариант с весом formula-SI `0.40`.

In [28]:
si_formula_weight_rows = []

si_formula_weight_grid = [0.0, 0.25, 0.4, 0.5, 0.6, 0.75, 1.0]

for si_formula_weight in si_formula_weight_grid:
    oof_predictions = make_mae_final_predictions(
        direct_predictions=mae_oof_direct_predictions,
        y_train=y,
        si_formula_weight=si_formula_weight,
        clip_quantile=0.995,
    )

    score = competition_rmse(y, oof_predictions)
    score_by_target = competition_rmse_by_target(
        y,
        oof_predictions,
        TARGET_COLUMNS,
    )

    si_formula_weight_rows.append(
        {
            "si_formula_weight": si_formula_weight,
            "oof_rmse": score,
            "oof_rmse_ic50": score_by_target["IC50"],
            "oof_rmse_cc50": score_by_target["CC50"],
            "oof_rmse_si": score_by_target["SI"],
        }
    )

si_formula_weight_results = pd.DataFrame(
    si_formula_weight_rows
).sort_values("oof_rmse")

display(si_formula_weight_results)

,si_formula_weight,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si
0,0.000000,527.929057,327.909456,470.236947,785.640770
6,1.000000,532.427423,327.909456,470.236947,799.135866
5,0.750000,532.514592,327.909456,470.236947,799.397373
4,0.600000,532.583000,327.909456,470.236947,799.602597
3,0.500000,532.635311,327.909456,470.236947,799.759530
2,0.400000,532.692982,327.909456,470.236947,799.932544
1,0.250000,532.789533,327.909456,470.236947,800.222198


### 4.7 Финальная CatBoost MAE ветка

Для дальнейшего объединения веток используется CatBoost MAE с параметрами:

- formula-SI weight: `0.40`;
- clipping quantile: `0.995`.

In [29]:
BEST_MAE_SI_FORMULA_WEIGHT = 0.40
MAE_CLIP_QUANTILE = 0.995

mae_best_oof_predictions = make_mae_final_predictions(
    direct_predictions=mae_oof_direct_predictions,
    y_train=y,
    si_formula_weight=BEST_MAE_SI_FORMULA_WEIGHT,
    clip_quantile=MAE_CLIP_QUANTILE,
)

mae_best_test_predictions = make_mae_final_predictions(
    direct_predictions=mae_test_direct_predictions,
    y_train=y,
    si_formula_weight=BEST_MAE_SI_FORMULA_WEIGHT,
    clip_quantile=MAE_CLIP_QUANTILE,
)

mae_best_oof_score = competition_rmse(y, mae_best_oof_predictions)
mae_best_oof_score_by_target = competition_rmse_by_target(
    y,
    mae_best_oof_predictions,
    TARGET_COLUMNS,
)

print(f"OOF RMSE CatBoost MAE best: {mae_best_oof_score:.6f}")
print("OOF RMSE по таргетам:")
print(mae_best_oof_score_by_target)

pd.DataFrame(
    mae_best_test_predictions,
    columns=TARGET_COLUMNS,
).describe().T

OOF RMSE CatBoost MAE best: 532.692982
OOF RMSE по таргетам:
IC50   327.909456
CC50   470.236947
SI     799.932544
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,177.956468,248.337404,0.542931,39.838308,87.421584,217.343309,1488.545439
CC50,250.000000,583.551342,457.336003,22.169839,270.928795,460.232612,788.050155,2857.502223
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


## 5. Объединение веток

После построения двух сильных веток выполняется их объединение:

1. `calibrated hybrid-best` — первая сильная ветка;
2. `CatBoost MAE` — вторая сильная ветка.

CatBoost MAE показал лучший standalone public score, но calibrated hybrid-best содержит полезный дополнительный сигнал для `IC50` и `CC50`.

Финальная логика объединения:

- `IC50`: blend CatBoost MAE и calibrated hybrid-best;
- `CC50`: blend CatBoost MAE и calibrated hybrid-best;
- `SI`: полностью остаётся из CatBoost MAE.

### 5.1 Подготовка двух веток

In [30]:
best_mae_si_formula_weight = 0.40

mae_best_oof_predictions = make_mae_final_predictions(
    direct_predictions=mae_oof_direct_predictions,
    y_train=y,
    si_formula_weight=best_mae_si_formula_weight,
    clip_quantile=0.995,
)

mae_best_test_predictions = make_mae_final_predictions(
    direct_predictions=mae_test_direct_predictions,
    y_train=y,
    si_formula_weight=best_mae_si_formula_weight,
    clip_quantile=0.995,
)

hybrid_best_ic50_weight = 0.075
hybrid_best_cc50_weight = 0.40

hybrid_best_oof_predictions = current_best_oof.copy()

hybrid_best_oof_predictions[:, 0] = (
    (1 - hybrid_best_ic50_weight) * current_best_oof[:, 0]
    + hybrid_best_ic50_weight * isotonic_branch_oof[:, 0]
)

hybrid_best_oof_predictions[:, 1] = (
    (1 - hybrid_best_cc50_weight) * current_best_oof[:, 1]
    + hybrid_best_cc50_weight * isotonic_branch_oof[:, 1]
)

hybrid_best_oof_predictions[:, 2] = current_best_oof[:, 2]
hybrid_best_oof_predictions[:, 2] = np.clip(
    hybrid_best_oof_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

hybrid_best_test_predictions = fine_meta_targetwise_predictions.copy()

hybrid_best_test_predictions[:, 0] = (
    (1 - hybrid_best_ic50_weight) * fine_meta_targetwise_predictions[:, 0]
    + hybrid_best_ic50_weight * isotonic_branch_test[:, 0]
)

hybrid_best_test_predictions[:, 1] = (
    (1 - hybrid_best_cc50_weight) * fine_meta_targetwise_predictions[:, 1]
    + hybrid_best_cc50_weight * isotonic_branch_test[:, 1]
)

hybrid_best_test_predictions[:, 2] = fine_meta_targetwise_predictions[:, 2]
hybrid_best_test_predictions[:, 2] = np.clip(
    hybrid_best_test_predictions[:, 2],
    SI_LOWER_BOUND,
    SI_UPPER_BOUND_Q0985,
)

branch_comparison_rows = []

for branch_name, branch_predictions in [
    ("catboost_mae", mae_best_oof_predictions),
    ("hybrid_best", hybrid_best_oof_predictions),
]:
    score = competition_rmse(y, branch_predictions)
    score_by_target = competition_rmse_by_target(
        y,
        branch_predictions,
        TARGET_COLUMNS,
    )

    branch_comparison_rows.append(
        {
            "branch": branch_name,
            "oof_rmse": score,
            "oof_rmse_ic50": score_by_target["IC50"],
            "oof_rmse_cc50": score_by_target["CC50"],
            "oof_rmse_si": score_by_target["SI"],
        }
    )

branch_comparison_df = pd.DataFrame(branch_comparison_rows)

display(branch_comparison_df)

,branch,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si
0,catboost_mae,532.692982,327.909456,470.236947,799.932544
1,hybrid_best,510.649345,339.228295,438.382996,754.336744


### 5.2 Локальная проверка blend двух веток

In [31]:
simple_branch_blend_rows = []

hybrid_weight_grid = [0.00, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20]

for hybrid_weight in hybrid_weight_grid:
    blended_predictions = (
        (1 - hybrid_weight) * mae_best_oof_predictions
        + hybrid_weight * hybrid_best_oof_predictions
    )

    blended_predictions[:, 2] = np.clip(
        blended_predictions[:, 2],
        SI_LOWER_BOUND,
        SI_UPPER_BOUND_Q0985,
    )

    score = competition_rmse(y, blended_predictions)
    score_by_target = competition_rmse_by_target(
        y,
        blended_predictions,
        TARGET_COLUMNS,
    )

    simple_branch_blend_rows.append(
        {
            "hybrid_weight": hybrid_weight,
            "mae_weight": 1 - hybrid_weight,
            "oof_rmse": score,
            "oof_rmse_ic50": score_by_target["IC50"],
            "oof_rmse_cc50": score_by_target["CC50"],
            "oof_rmse_si": score_by_target["SI"],
        }
    )

simple_branch_blend_results = pd.DataFrame(
    simple_branch_blend_rows
).sort_values("oof_rmse")

display(simple_branch_blend_results)

,hybrid_weight,mae_weight,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si
6,0.200000,0.800000,520.632763,324.282847,459.836973,777.778470
5,0.150000,0.850000,522.243653,324.913854,462.262004,779.555103
4,0.100000,0.900000,523.968022,325.729596,464.804966,781.369505
3,0.075000,0.925000,524.872394,326.206321,466.120072,782.290787
2,0.050000,0.950000,525.804680,326.728688,467.463937,783.221414
1,0.025000,0.975000,526.764713,327.296478,468.836311,784.161351
0,0.000000,1.000000,527.752322,327.909456,470.236947,785.110563


### 5.3 Локальная проверка target-wise blend

In [32]:
targetwise_branch_blend_rows = []

ic50_hybrid_weight_grid = [0.00, 0.025, 0.05, 0.075, 0.10]
cc50_hybrid_weight_grid = [0.00, 0.025, 0.05, 0.075, 0.10]
si_hybrid_weight_grid = [0.00, 0.025, 0.05, 0.075, 0.10, 0.15]

for ic50_hybrid_weight in ic50_hybrid_weight_grid:
    for cc50_hybrid_weight in cc50_hybrid_weight_grid:
        for si_hybrid_weight in si_hybrid_weight_grid:
            blended_predictions = mae_best_oof_predictions.copy()

            blended_predictions[:, 0] = (
                (1 - ic50_hybrid_weight) * mae_best_oof_predictions[:, 0]
                + ic50_hybrid_weight * hybrid_best_oof_predictions[:, 0]
            )

            blended_predictions[:, 1] = (
                (1 - cc50_hybrid_weight) * mae_best_oof_predictions[:, 1]
                + cc50_hybrid_weight * hybrid_best_oof_predictions[:, 1]
            )

            blended_predictions[:, 2] = (
                (1 - si_hybrid_weight) * mae_best_oof_predictions[:, 2]
                + si_hybrid_weight * hybrid_best_oof_predictions[:, 2]
            )

            blended_predictions[:, 2] = np.clip(
                blended_predictions[:, 2],
                SI_LOWER_BOUND,
                SI_UPPER_BOUND_Q0985,
            )

            score = competition_rmse(y, blended_predictions)
            score_by_target = competition_rmse_by_target(
                y,
                blended_predictions,
                TARGET_COLUMNS,
            )

            targetwise_branch_blend_rows.append(
                {
                    "ic50_hybrid_weight": ic50_hybrid_weight,
                    "cc50_hybrid_weight": cc50_hybrid_weight,
                    "si_hybrid_weight": si_hybrid_weight,
                    "oof_rmse": score,
                    "oof_rmse_ic50": score_by_target["IC50"],
                    "oof_rmse_cc50": score_by_target["CC50"],
                    "oof_rmse_si": score_by_target["SI"],
                }
            )

targetwise_branch_blend_results = pd.DataFrame(
    targetwise_branch_blend_rows
).sort_values("oof_rmse")

display(targetwise_branch_blend_results.head(30))

,ic50_hybrid_weight,cc50_hybrid_weight,si_hybrid_weight,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si
149,0.100000,0.100000,0.150000,523.363222,325.729596,464.804966,779.555103
119,0.075000,0.100000,0.150000,523.522130,326.206321,464.804966,779.555103
89,0.050000,0.100000,0.150000,523.696252,326.728688,464.804966,779.555103
143,0.100000,0.075000,0.150000,523.801590,325.729596,466.120072,779.555103
59,0.025000,0.100000,0.150000,523.885516,327.296478,464.804966,779.555103
113,0.075000,0.075000,0.150000,523.960499,326.206321,466.120072,779.555103
148,0.100000,0.100000,0.100000,523.968022,325.729596,464.804966,781.369505
29,0.000000,0.100000,0.150000,524.089842,327.909456,464.804966,779.555103
118,0.075000,0.100000,0.100000,524.126931,326.206321,464.804966,781.369505
83,0.050000,0.075000,0.150000,524.134621,326.728688,466.120072,779.555103


### 5.4 Финальная функция blend для `IC50` и `CC50`

После проверки target-wise blend итоговая схема фиксирует `SI` из CatBoost MAE и смешивает только `IC50` и `CC50`.

Несмотря на то, что локальная проверка допускала небольшой вес hybrid-best для `SI`, в исследовательском ноутбуке такие варианты не дали улучшения на public leaderboard. Поэтому в финальном pipeline `SI` не смешивается и полностью остаётся из CatBoost MAE ветки.

In [33]:
def make_ic50_cc50_blend_predictions(
    mae_predictions,
    hybrid_predictions,
    ic50_hybrid_weight,
    cc50_hybrid_weight,
):
    """Смешивает только IC50 и CC50, оставляя SI из CatBoost MAE ветки."""
    blend_predictions = mae_predictions.copy()

    blend_predictions[:, 0] = (
        (1 - ic50_hybrid_weight) * mae_predictions[:, 0]
        + ic50_hybrid_weight * hybrid_predictions[:, 0]
    )

    blend_predictions[:, 1] = (
        (1 - cc50_hybrid_weight) * mae_predictions[:, 1]
        + cc50_hybrid_weight * hybrid_predictions[:, 1]
    )

    blend_predictions[:, 2] = mae_predictions[:, 2]

    return blend_predictions

### 5.5 Локальная проверка одинаковых весов для `IC50` и `CC50`

In [34]:
local_weight_grid = np.round(np.arange(0.0, 1.001, 0.025), 3)

local_blend_rows = []

for hybrid_weight in local_weight_grid:
    oof_predictions = make_ic50_cc50_blend_predictions(
        mae_predictions=mae_best_oof_predictions,
        hybrid_predictions=hybrid_best_oof_predictions,
        ic50_hybrid_weight=hybrid_weight,
        cc50_hybrid_weight=hybrid_weight,
    )

    test_predictions = make_ic50_cc50_blend_predictions(
        mae_predictions=mae_best_test_predictions,
        hybrid_predictions=hybrid_best_test_predictions,
        ic50_hybrid_weight=hybrid_weight,
        cc50_hybrid_weight=hybrid_weight,
    )

    score = competition_rmse(y, oof_predictions)
    score_by_target = competition_rmse_by_target(
        y,
        oof_predictions,
        TARGET_COLUMNS,
    )

    test_summary = pd.DataFrame(
        test_predictions,
        columns=TARGET_COLUMNS,
    ).describe().T

    local_blend_rows.append(
        {
            "hybrid_weight": hybrid_weight,
            "mae_weight": 1 - hybrid_weight,
            "oof_rmse": score,
            "oof_rmse_ic50": score_by_target["IC50"],
            "oof_rmse_cc50": score_by_target["CC50"],
            "oof_rmse_si": score_by_target["SI"],
            "test_ic50_mean": test_summary.loc["IC50", "mean"],
            "test_ic50_std": test_summary.loc["IC50", "std"],
            "test_ic50_max": test_summary.loc["IC50", "max"],
            "test_cc50_mean": test_summary.loc["CC50", "mean"],
            "test_cc50_std": test_summary.loc["CC50", "std"],
            "test_cc50_max": test_summary.loc["CC50", "max"],
            "test_si_mean": test_summary.loc["SI", "mean"],
            "test_si_std": test_summary.loc["SI", "std"],
            "test_si_max": test_summary.loc["SI", "max"],
        }
    )

local_blend_results = pd.DataFrame(local_blend_rows)

display(
    local_blend_results
    .sort_values("oof_rmse")
    .head(20)
)

print("Зона вокруг текущего лучшего public-веса:")
display(
    local_blend_results[
        local_blend_results["hybrid_weight"].between(0.35, 0.70)
    ]
)

,hybrid_weight,mae_weight,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si,test_ic50_mean,test_ic50_std,test_ic50_max,test_cc50_mean,test_cc50_std,test_cc50_max,test_si_mean,test_si_std,test_si_max
27,0.675000,0.325000,523.508255,327.579038,443.013184,799.932544,225.125093,296.428167,1987.630451,626.628100,493.081974,3242.362354,14.467613,28.665825,251.911744
26,0.650000,0.350000,523.509783,326.989974,443.606830,799.932544,223.378107,294.413048,1969.145821,625.032664,491.548388,3228.108275,14.467613,28.665825,251.911744
28,0.700000,0.300000,523.532917,328.213172,442.453035,799.932544,226.872079,298.458071,2006.115081,628.223535,494.630381,3256.616432,14.467613,28.665825,251.911744
25,0.625000,0.375000,523.537535,326.446224,444.233837,799.932544,221.631121,292.413021,1950.661191,623.437229,490.029760,3213.854196,14.467613,28.665825,251.911744
29,0.725000,0.275000,523.583723,328.892115,441.926509,799.932544,228.619066,300.502462,2024.599711,629.818971,496.193468,3270.870511,14.467613,28.665825,251.911744
24,0.600000,0.400000,523.591542,325.948016,444.894066,799.932544,219.884135,290.428397,1932.176561,621.841793,488.526230,3199.600117,14.467613,28.665825,251.911744
30,0.750000,0.250000,523.660621,329.615592,441.433727,799.932544,230.366052,302.561045,2043.084341,631.414406,497.771098,3285.124590,14.467613,28.665825,251.911744
23,0.575000,0.425000,523.671823,325.495557,445.587367,799.932544,218.137149,288.459494,1913.691931,620.246358,487.037939,3185.346038,14.467613,28.665825,251.911744
31,0.775000,0.225000,523.763551,330.383308,440.974802,799.932544,232.113038,304.633533,2061.568972,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
22,0.550000,0.450000,523.778391,325.089040,446.313588,799.932544,216.390163,286.506636,1895.207301,618.650922,485.565026,3171.091959,14.467613,28.665825,251.911744


Зона вокруг текущего лучшего public-веса:


,hybrid_weight,mae_weight,oof_rmse,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si,test_ic50_mean,test_ic50_std,test_ic50_max,test_cc50_mean,test_cc50_std,test_cc50_max,test_si_mean,test_si_std,test_si_max
14,0.350000,0.650000,525.575982,323.507458,453.287944,799.932544,202.414274,271.502359,1747.330260,605.887438,474.352221,3057.059328,14.467613,28.665825,251.911744
15,0.375000,0.625000,525.259679,323.541854,452.304639,799.932544,204.161260,273.314437,1765.814890,607.482874,475.697042,3071.313407,14.467613,28.665825,251.911744
16,0.400000,0.600000,524.969460,323.623027,451.352807,799.932544,205.908246,275.145378,1784.299520,609.078309,477.058367,3085.567486,14.467613,28.665825,251.911744
17,0.425000,0.575000,524.705380,323.750944,450.432650,799.932544,207.655232,276.994808,1802.784150,610.673745,478.436057,3099.821564,14.467613,28.665825,251.911744
18,0.450000,0.550000,524.467485,323.925549,449.544360,799.932544,209.402218,278.862359,1821.268781,612.269180,479.829969,3114.075643,14.467613,28.665825,251.911744
19,0.475000,0.525000,524.255813,324.146766,448.688128,799.932544,211.149204,280.747670,1839.753411,613.864616,481.239963,3128.329722,14.467613,28.665825,251.911744
20,0.500000,0.500000,524.070394,324.414501,447.864138,799.932544,212.896191,282.650384,1858.238041,615.460051,482.665897,3142.583801,14.467613,28.665825,251.911744
21,0.525000,0.475000,523.911249,324.728637,447.072567,799.932544,214.643177,284.570154,1876.722671,617.055487,484.107632,3156.837880,14.467613,28.665825,251.911744
22,0.550000,0.450000,523.778391,325.089040,446.313588,799.932544,216.390163,286.506636,1895.207301,618.650922,485.565026,3171.091959,14.467613,28.665825,251.911744
23,0.575000,0.425000,523.671823,325.495557,445.587367,799.932544,218.137149,288.459494,1913.691931,620.246358,487.037939,3185.346038,14.467613,28.665825,251.911744


### 5.6 Локальная проверка асимметричных весов

In [35]:
ic50_weight_grid = [0.55, 0.60, 0.65, 0.675, 0.70, 0.725, 0.75]
cc50_weight_grid = [0.65, 0.675, 0.70, 0.725, 0.75, 0.775, 0.80]

current_best_ic50_weight = 0.725
current_best_cc50_weight = 0.725

current_best_oof_predictions_asymmetric = make_ic50_cc50_blend_predictions(
    mae_predictions=mae_best_oof_predictions,
    hybrid_predictions=hybrid_best_oof_predictions,
    ic50_hybrid_weight=current_best_ic50_weight,
    cc50_hybrid_weight=current_best_cc50_weight,
)

current_best_oof_score = competition_rmse(
    y,
    current_best_oof_predictions_asymmetric,
)

asymmetric_blend_rows = []

for ic50_hybrid_weight in ic50_weight_grid:
    for cc50_hybrid_weight in cc50_weight_grid:
        oof_predictions = make_ic50_cc50_blend_predictions(
            mae_predictions=mae_best_oof_predictions,
            hybrid_predictions=hybrid_best_oof_predictions,
            ic50_hybrid_weight=ic50_hybrid_weight,
            cc50_hybrid_weight=cc50_hybrid_weight,
        )

        test_predictions = make_ic50_cc50_blend_predictions(
            mae_predictions=mae_best_test_predictions,
            hybrid_predictions=hybrid_best_test_predictions,
            ic50_hybrid_weight=ic50_hybrid_weight,
            cc50_hybrid_weight=cc50_hybrid_weight,
        )

        score = competition_rmse(y, oof_predictions)
        score_by_target = competition_rmse_by_target(
            y,
            oof_predictions,
            TARGET_COLUMNS,
        )

        test_summary = pd.DataFrame(
            test_predictions,
            columns=TARGET_COLUMNS,
        ).describe().T

        asymmetric_blend_rows.append(
            {
                "ic50_hybrid_weight": ic50_hybrid_weight,
                "cc50_hybrid_weight": cc50_hybrid_weight,
                "oof_rmse": score,
                "oof_delta_from_0725_0725": (
                    score - current_best_oof_score
                ),
                "oof_rmse_ic50": score_by_target["IC50"],
                "oof_rmse_cc50": score_by_target["CC50"],
                "oof_rmse_si": score_by_target["SI"],
                "test_ic50_mean": test_summary.loc["IC50", "mean"],
                "test_ic50_std": test_summary.loc["IC50", "std"],
                "test_ic50_max": test_summary.loc["IC50", "max"],
                "test_cc50_mean": test_summary.loc["CC50", "mean"],
                "test_cc50_std": test_summary.loc["CC50", "std"],
                "test_cc50_max": test_summary.loc["CC50", "max"],
                "test_si_mean": test_summary.loc["SI", "mean"],
                "test_si_std": test_summary.loc["SI", "std"],
                "test_si_max": test_summary.loc["SI", "max"],
            }
        )

asymmetric_blend_results = pd.DataFrame(asymmetric_blend_rows)

print("Лучшие варианты по OOF RMSE:")
display(
    asymmetric_blend_results
    .sort_values("oof_rmse")
    .head(20)
)

print("Варианты рядом с текущим лучшим public-весом 0.725 / 0.725:")
display(
    asymmetric_blend_results[
        (
            asymmetric_blend_results["ic50_hybrid_weight"]
            .between(0.65, 0.75)
        )
        & (
            asymmetric_blend_results["cc50_hybrid_weight"]
            .between(0.70, 0.80)
        )
    ].sort_values("oof_rmse")
)

Лучшие варианты по OOF RMSE:


,ic50_hybrid_weight,cc50_hybrid_weight,oof_rmse,oof_delta_from_0725_0725,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si,test_ic50_mean,test_ic50_std,test_ic50_max,test_cc50_mean,test_cc50_std,test_cc50_max,test_si_mean,test_si_std,test_si_max
6,0.550000,0.800000,521.857141,-1.726582,325.089040,440.549839,799.932544,216.390163,286.506636,1895.207301,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
5,0.550000,0.775000,521.998795,-1.584927,325.089040,440.974802,799.932544,216.390163,286.506636,1895.207301,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
13,0.600000,0.800000,522.143467,-1.440256,325.948016,440.549839,799.932544,219.884135,290.428397,1932.176561,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
4,0.550000,0.750000,522.151770,-1.431952,325.089040,441.433727,799.932544,216.390163,286.506636,1895.207301,631.414406,497.771098,3285.124590,14.467613,28.665825,251.911744
12,0.600000,0.775000,522.285121,-1.298602,325.948016,440.974802,799.932544,219.884135,290.428397,1932.176561,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
3,0.550000,0.725000,522.316031,-1.267692,325.089040,441.926509,799.932544,216.390163,286.506636,1895.207301,629.818971,496.193468,3270.870511,14.467613,28.665825,251.911744
11,0.600000,0.750000,522.438096,-1.145627,325.948016,441.433727,799.932544,219.884135,290.428397,1932.176561,631.414406,497.771098,3285.124590,14.467613,28.665825,251.911744
20,0.650000,0.800000,522.490786,-1.092937,326.989974,440.549839,799.932544,223.378107,294.413048,1969.145821,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
2,0.550000,0.700000,522.491540,-1.092183,325.089040,442.453035,799.932544,216.390163,286.506636,1895.207301,628.223535,494.630381,3256.616432,14.467613,28.665825,251.911744
10,0.600000,0.725000,522.602356,-0.981367,325.948016,441.926509,799.932544,219.884135,290.428397,1932.176561,629.818971,496.193468,3270.870511,14.467613,28.665825,251.911744


Варианты рядом с текущим лучшим public-весом 0.725 / 0.725:


,ic50_hybrid_weight,cc50_hybrid_weight,oof_rmse,oof_delta_from_0725_0725,oof_rmse_ic50,oof_rmse_cc50,oof_rmse_si,test_ic50_mean,test_ic50_std,test_ic50_max,test_cc50_mean,test_cc50_std,test_cc50_max,test_si_mean,test_si_std,test_si_max
20,0.650000,0.800000,522.490786,-1.092937,326.989974,440.549839,799.932544,223.378107,294.413048,1969.145821,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
19,0.650000,0.775000,522.632440,-0.951283,326.989974,440.974802,799.932544,223.378107,294.413048,1969.145821,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
27,0.675000,0.800000,522.687140,-0.896582,327.579038,440.549839,799.932544,225.125093,296.428167,1987.630451,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
18,0.650000,0.750000,522.785415,-0.798308,326.989974,441.433727,799.932544,223.378107,294.413048,1969.145821,631.414406,497.771098,3285.124590,14.467613,28.665825,251.911744
26,0.675000,0.775000,522.828795,-0.754928,327.579038,440.974802,799.932544,225.125093,296.428167,1987.630451,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
34,0.700000,0.800000,522.898519,-0.685204,328.213172,440.549839,799.932544,226.872079,298.458071,2006.115081,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744
17,0.650000,0.725000,522.949676,-0.634047,326.989974,441.926509,799.932544,223.378107,294.413048,1969.145821,629.818971,496.193468,3270.870511,14.467613,28.665825,251.911744
25,0.675000,0.750000,522.981770,-0.601953,327.579038,441.433727,799.932544,225.125093,296.428167,1987.630451,631.414406,497.771098,3285.124590,14.467613,28.665825,251.911744
33,0.700000,0.775000,523.040173,-0.543550,328.213172,440.974802,799.932544,226.872079,298.458071,2006.115081,633.009842,499.363133,3299.378669,14.467613,28.665825,251.911744
41,0.725000,0.800000,523.124833,-0.458890,328.892115,440.549839,799.932544,228.619066,300.502462,2024.599711,634.605277,500.969436,3313.632748,14.467613,28.665825,251.911744


### 5.7 Финальная объединённая ветка

По результатам исследовательского ноутбука для финального pipeline выбран симметричный blend:

- `IC50`: 72.5% calibrated hybrid-best и 27.5% CatBoost MAE;
- `CC50`: 72.5% calibrated hybrid-best и 27.5% CatBoost MAE;
- `SI`: 100% CatBoost MAE.

Асимметричные веса локально давали более низкий OOF RMSE, но не улучшили public score в исследовательском ноутбуке. Поэтому для финального pipeline оставлен симметричный вариант `0.725 / 0.725`.

Локальная проверка давала близкие значения в диапазоне примерно `0.65–0.725`. Финально выбран вес `0.725`, потому что именно он был зафиксирован в исследовательском ноутбуке как лучший public-кандидат. Локальная OOF-метрика использовалась как ориентир, но не как единственный критерий выбора между близкими вариантами.

In [36]:
best_ic50_weight = 0.725
best_cc50_weight = 0.725

current_best_oof_predictions = make_ic50_cc50_blend_predictions(
    mae_predictions=mae_best_oof_predictions,
    hybrid_predictions=hybrid_best_oof_predictions,
    ic50_hybrid_weight=best_ic50_weight,
    cc50_hybrid_weight=best_cc50_weight,
)

current_best_test_predictions = make_ic50_cc50_blend_predictions(
    mae_predictions=mae_best_test_predictions,
    hybrid_predictions=hybrid_best_test_predictions,
    ic50_hybrid_weight=best_ic50_weight,
    cc50_hybrid_weight=best_cc50_weight,
)

current_best_oof_score = competition_rmse(
    y,
    current_best_oof_predictions,
)

current_best_oof_score_by_target = competition_rmse_by_target(
    y,
    current_best_oof_predictions,
    TARGET_COLUMNS,
)

print(f"OOF RMSE объединённой ветки: {current_best_oof_score:.6f}")
print("OOF RMSE по таргетам:")
print(current_best_oof_score_by_target)

pd.DataFrame(
    current_best_test_predictions,
    columns=TARGET_COLUMNS,
).describe().T

OOF RMSE объединённой ветки: 523.583723
OOF RMSE по таргетам:
IC50   328.892115
CC50   441.926509
SI     799.932544
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,228.619066,300.502462,2.911020,58.955089,121.859591,270.523259,2024.599711
CC50,250.000000,629.818971,496.193468,25.596459,303.670671,510.714727,813.097856,3270.870511
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


## 6. Postprocessing `IC50`

После объединения CatBoost MAE и calibrated hybrid-best применяется финальный postprocessing только для `IC50`.

В `01_research_log` были найдены две проблемные зоны:

1. **Low-IC50 зона** — текущий blend завышал малые значения `IC50`.
2. **High-IC50 зона** — текущий blend занижал большие значения `IC50`.

Поэтому в финальном pipeline используются две коррекции:

- low-IC50 correction: если CatBoost MAE предсказывает `IC50 <= 50`, итоговый `IC50` сдвигается в сторону CatBoost MAE с весом `0.75`;
- high-IC50 uplift: если текущий прогноз `IC50 >= 300`, значение `IC50` умножается на `1.10`.

`CC50` и `SI` в postprocessing не меняются.

### 6.1 Low-IC50 correction

In [37]:
def apply_low_ic50_correction(
    base_predictions,
    mae_predictions,
    low_ic50_threshold,
    mae_weight_for_low_zone,
):
    """Корректирует IC50 в зоне low-IC50 по маске из CatBoost MAE прогноза."""
    corrected_predictions = base_predictions.copy()

    low_ic50_mask = mae_predictions[:, 0] <= low_ic50_threshold

    corrected_predictions[low_ic50_mask, 0] = (
        (1 - mae_weight_for_low_zone) * base_predictions[low_ic50_mask, 0]
        + mae_weight_for_low_zone * mae_predictions[low_ic50_mask, 0]
    )

    return corrected_predictions, low_ic50_mask

### 6.2 Применение low-IC50 correction

Финальные параметры из `01_research_log`:

- `LOW_IC50_THRESHOLD = 50`;
- `LOW_IC50_MAE_WEIGHT = 0.75`.

Перед применением correction заново фиксируется базовая объединённая ветка с весами `0.725 / 0.725`.

In [38]:
CURRENT_BEST_IC50_WEIGHT = 0.725
CURRENT_BEST_CC50_WEIGHT = 0.725
LOW_IC50_THRESHOLD = 50
LOW_IC50_MAE_WEIGHT = 0.75

base_best_oof_predictions = make_ic50_cc50_blend_predictions(
    mae_predictions=mae_best_oof_predictions,
    hybrid_predictions=hybrid_best_oof_predictions,
    ic50_hybrid_weight=CURRENT_BEST_IC50_WEIGHT,
    cc50_hybrid_weight=CURRENT_BEST_CC50_WEIGHT,
)

base_best_test_predictions = make_ic50_cc50_blend_predictions(
    mae_predictions=mae_best_test_predictions,
    hybrid_predictions=hybrid_best_test_predictions,
    ic50_hybrid_weight=CURRENT_BEST_IC50_WEIGHT,
    cc50_hybrid_weight=CURRENT_BEST_CC50_WEIGHT,
)

current_best_oof_predictions, current_best_low_ic50_oof_mask = (
    apply_low_ic50_correction(
        base_predictions=base_best_oof_predictions,
        mae_predictions=mae_best_oof_predictions,
        low_ic50_threshold=LOW_IC50_THRESHOLD,
        mae_weight_for_low_zone=LOW_IC50_MAE_WEIGHT,
    )
)

current_best_test_predictions, current_best_low_ic50_test_mask = (
    apply_low_ic50_correction(
        base_predictions=base_best_test_predictions,
        mae_predictions=mae_best_test_predictions,
        low_ic50_threshold=LOW_IC50_THRESHOLD,
        mae_weight_for_low_zone=LOW_IC50_MAE_WEIGHT,
    )
)

current_best_score = competition_rmse(
    y,
    current_best_oof_predictions,
)

current_best_score_by_target = competition_rmse_by_target(
    y,
    current_best_oof_predictions,
    TARGET_COLUMNS,
)

print("OOF RMSE текущего решения после low-IC50 correction:")
print(f"{current_best_score:.6f}")
print("OOF RMSE по таргетам:")
print(current_best_score_by_target)

print("Количество OOF-объектов с low-IC50 correction:")
print(int(current_best_low_ic50_oof_mask.sum()))

print("Количество test-объектов с low-IC50 correction:")
print(int(current_best_low_ic50_test_mask.sum()))

pd.DataFrame(
    current_best_test_predictions,
    columns=TARGET_COLUMNS,
).describe().T

OOF RMSE текущего решения после low-IC50 correction:
521.662452
OOF RMSE по таргетам:
IC50   323.128303
CC50   441.926509
SI     799.932544
dtype: float64
Количество OOF-объектов с low-IC50 correction:
253
Количество test-объектов с low-IC50 correction:
81


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,224.542809,302.808784,1.134953,45.950877,121.489468,270.523259,2024.599711
CC50,250.000000,629.818971,496.193468,25.596459,303.670671,510.714727,813.097856,3270.870511
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


### 6.3 High-IC50 uplift

In [39]:
def apply_high_ic50_uplift(
    base_predictions,
    mask,
    uplift_factor,
    upper_limit=None,
):
    """Увеличивает IC50 для объектов, выбранных маской."""
    uplifted_predictions = base_predictions.copy()

    uplifted_predictions[mask, 0] = (
        uplifted_predictions[mask, 0] * uplift_factor
    )

    if upper_limit is not None:
        uplifted_predictions[:, 0] = np.clip(
            uplifted_predictions[:, 0],
            0.001,
            upper_limit,
        )

    return uplifted_predictions

### 6.4 Финальный high-IC50 uplift

Финальные параметры из `01_research_log`:

- `HIGH_IC50_THRESHOLD = 300`;
- `HIGH_IC50_UPLIFT_FACTOR = 1.10`.

Маска строится по текущему прогнозу после low-IC50 correction. Коррекция применяется только к `IC50`.

In [40]:
HIGH_IC50_THRESHOLD = 300
HIGH_IC50_UPLIFT_FACTOR = 1.10

high_ic50_oof_mask = current_best_oof_predictions[:, 0] >= HIGH_IC50_THRESHOLD
high_ic50_test_mask = current_best_test_predictions[:, 0] >= HIGH_IC50_THRESHOLD

postprocessed_best_oof_predictions = apply_high_ic50_uplift(
    base_predictions=current_best_oof_predictions,
    mask=high_ic50_oof_mask,
    uplift_factor=HIGH_IC50_UPLIFT_FACTOR,
    upper_limit=y["IC50"].max(),
)

postprocessed_best_test_predictions = apply_high_ic50_uplift(
    base_predictions=current_best_test_predictions,
    mask=high_ic50_test_mask,
    uplift_factor=HIGH_IC50_UPLIFT_FACTOR,
    upper_limit=y["IC50"].max(),
)

postprocessed_best_oof_score = competition_rmse(
    y,
    postprocessed_best_oof_predictions,
)

postprocessed_best_oof_by_target = competition_rmse_by_target(
    y,
    postprocessed_best_oof_predictions,
    TARGET_COLUMNS,
)

print(f"OOF RMSE после финального postprocessing: {postprocessed_best_oof_score:.6f}")
print("OOF RMSE по таргетам:")
print(postprocessed_best_oof_by_target)

print(f"Количество изменённых OOF-объектов: {int(high_ic50_oof_mask.sum())}")
print(f"Количество изменённых test-объектов: {int(high_ic50_test_mask.sum())}")

pd.DataFrame(
    postprocessed_best_test_predictions,
    columns=TARGET_COLUMNS,
).describe().T

OOF RMSE после финального postprocessing: 521.627878
OOF RMSE по таргетам:
IC50   323.024581
CC50   441.926509
SI     799.932544
dtype: float64
Количество изменённых OOF-объектов: 138
Количество изменённых test-объектов: 56


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,238.922243,334.798653,1.134953,45.950877,121.489468,270.523259,2227.059683
CC50,250.000000,629.818971,496.193468,25.596459,303.670671,510.714727,813.097856,3270.870511
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


### 6.5 Финальная фиксация предсказаний

После postprocessing финальными предсказаниями считаются `postprocessed_best_oof_predictions` и `postprocessed_best_test_predictions`.

Обе postprocessing-коррекции меняют только `IC50`. `CC50` и `SI` остаются из объединённой ветки.

In [41]:
final_oof_predictions = postprocessed_best_oof_predictions.copy()
final_test_predictions = postprocessed_best_test_predictions.copy()

final_oof_score = competition_rmse(
    y,
    final_oof_predictions,
)

final_oof_score_by_target = competition_rmse_by_target(
    y,
    final_oof_predictions,
    TARGET_COLUMNS,
)

print(f"Final OOF RMSE: {final_oof_score:.6f}")
print("Final OOF RMSE по таргетам:")
print(final_oof_score_by_target)

validate_predictions(
    predictions=final_test_predictions,
    expected_rows=len(sample_submission),
    target_columns=TARGET_COLUMNS,
)

describe_predictions(final_test_predictions, TARGET_COLUMNS)

Final OOF RMSE: 521.627878
Final OOF RMSE по таргетам:
IC50   323.024581
CC50   441.926509
SI     799.932544
dtype: float64


,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,238.922243,334.798653,1.134953,45.950877,121.489468,270.523259,2227.059683
CC50,250.000000,629.818971,496.193468,25.596459,303.670671,510.714727,813.097856,3270.870511
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


## 7. Создание финального submission

В этом разделе создаётся финальный файл с предсказаниями для Kaggle.

Финальные предсказания берутся из переменной `final_test_predictions`, которая получена после:

1. построения calibrated hybrid-best ветки;
2. построения CatBoost MAE ветки;
3. объединения веток для `IC50` и `CC50`;
4. postprocessing только для `IC50`.

`CC50` и `SI` после объединения веток дополнительно не изменяются.

### 7.1 Проверка финальных предсказаний

In [42]:
validate_predictions(
    predictions=final_test_predictions,
    expected_rows=len(sample_submission),
    target_columns=TARGET_COLUMNS,
)

final_test_summary = describe_predictions(
    predictions=final_test_predictions,
    target_columns=TARGET_COLUMNS,
)

final_test_summary

,count,mean,std,min,25%,50%,75%,max
IC50,250.000000,238.922243,334.798653,1.134953,45.950877,121.489468,270.523259,2227.059683
CC50,250.000000,629.818971,496.193468,25.596459,303.670671,510.714727,813.097856,3270.870511
SI,250.000000,14.467613,28.665825,1.333562,3.867970,6.455990,13.917248,251.911744


### 7.2 Создание submission-файла

In [43]:
FINAL_SUBMISSION_PATH = SUBMISSIONS_DIR / "final_submission.csv"

final_submission = create_submission(
    predictions=final_test_predictions,
    sample_submission=sample_submission,
    output_path=FINAL_SUBMISSION_PATH,
)

final_submission.head()

Submission сохранён: ../submissions/final_submission.csv
Размер submission: (250, 4)


,index,IC50,CC50,SI
0,0,106.969832,350.920911,3.823805
1,1,130.032171,380.916020,4.665967
2,2,52.072606,366.882005,5.723671
3,3,257.678391,471.612118,3.310459
4,4,195.023217,371.382765,2.417726


### 7.3 Финальная проверка формата

In [44]:
print("Финальная проверка submission")
print(f"Путь к файлу: {FINAL_SUBMISSION_PATH}")
print(f"Размер: {final_submission.shape}")
print(f"Колонки: {final_submission.columns.tolist()}")

if list(final_submission.columns) != [ID_COLUMN] + TARGET_COLUMNS:
    raise ValueError("Некорректный порядок колонок в submission.")

if final_submission[TARGET_COLUMNS].isna().any().any():
    raise ValueError("В submission есть NaN.")

if np.isinf(final_submission[TARGET_COLUMNS].to_numpy()).any():
    raise ValueError("В submission есть бесконечные значения.")

if final_submission[ID_COLUMN].duplicated().any():
    raise ValueError("В submission есть дублирующиеся index.")

print("Формат submission корректный.")

Финальная проверка submission
Путь к файлу: ../submissions/final_submission.csv
Размер: (250, 4)
Колонки: ['index', 'IC50', 'CC50', 'SI']
Формат submission корректный.


### 7.4 Итог

Финальный submission сохранён в файл:

`../submissions/final_submission.csv`

Итоговая схема решения:
- IC50: blend calibrated hybrid-best и CatBoost MAE + low/high postprocessing;
- CC50: blend calibrated hybrid-best и CatBoost MAE;
- SI: CatBoost MAE direct/model-прогноз.

Финальный public score этого решения: 263.61199.